
# Evaluación del reconocimiento facial con personas no registradas

## Objetivo

Evaluar la capacidad del sistema de reconocimiento facial para clasificar como desconocidos los rostros de personas que no pertenecen a la galería de referencia.

## Metodología

Se utilizaron 10 fotografías almacenadas en la carpeta `Unknown`, correspondientes a personas no registradas en el sistema.

La evaluación se realizó utilizando el modelo previamente configurado y las codificaciones faciales almacenadas en `encodings.pickle`, sin modificar la galería original.

Los parámetros utilizados fueron:

- Detector facial: HOG.
- Umbral de reconocimiento: 0.48.
- Coincidencias más cercanas: 3.
- Margen mínimo de clasificación: 0.055.

Se analizó la cantidad de rostros detectados y las etiquetas asignadas por el sistema, con el propósito de identificar posibles clasificaciones incorrectas.


In [18]:
import cv2
import pickle
import numpy as np
import pandas as pd
import face_recognition
import matplotlib.pyplot as plt
from pathlib import Path

# CONFIGURACIÓN
CARPETA = Path("Unknown")
ENTRENAMIENTO = "encodings.pickle"

UMBRAL = 0.48
NEAREST = 3
MARGEN = 0.055

# Cargar la galería facial existente
with open(ENTRENAMIENTO, "rb") as f:
    datos = pickle.load(f)

nombres = np.asarray(datos["names"])
conocidos = np.asarray(datos["encodings"])

personas = sorted(set(nombres))

# Buscar fotografías
extensiones = {".jpg", ".jpeg", ".png", ".bmp"}

fotos = sorted(
    p for p in CARPETA.rglob("*")
    if p.is_file() and p.suffix.lower() in extensiones
)

resultados = []
sin_rostro = []
no_leidas = []

for foto in fotos:

    imagen = cv2.imread(str(foto))

    if imagen is None:
        no_leidas.append(foto.name)
        continue

    rgb = cv2.cvtColor(imagen, cv2.COLOR_BGR2RGB)

    ubicaciones = face_recognition.face_locations(
        rgb, model="hog"
    )

    codificaciones = face_recognition.face_encodings(
        rgb, ubicaciones
    )

    if len(codificaciones) == 0:
        sin_rostro.append(foto.name)

    for i, encoding in enumerate(codificaciones):

        distancias = face_recognition.face_distance(
            conocidos, encoding
        )

        puntuaciones = {}

        for persona in personas:
            valores = np.sort(
                distancias[nombres == persona]
            )

            puntuaciones[persona] = float(
                np.mean(valores[:NEAREST])
            )

        ordenados = sorted(
            puntuaciones.items(),
            key=lambda x: x[1]
        )

        mejor_nombre, mejor_distancia = ordenados[0]

        segunda_distancia = (
            ordenados[1][1]
            if len(ordenados) > 1
            else float("inf")
        )

        diferencia = segunda_distancia - mejor_distancia

        if (
            mejor_distancia <= UMBRAL
            and diferencia >= MARGEN
        ):
            prediccion = mejor_nombre
        else:
            prediccion = "Desconocido"

        resultados.append({
            "Fotografia": foto.name,
            "Rostro": i + 1,
            "Prediccion": prediccion,
            "Distancia": round(mejor_distancia, 4),
            "Margen": round(diferencia, 4)
        })

# Mostrar los resultados
tabla = pd.DataFrame(resultados)

print("FOTOGRAFÍAS ANALIZADAS:", len(fotos))
print("FOTOGRAFÍAS NO LEÍDAS:", len(no_leidas))
print("FOTOGRAFÍAS SIN ROSTROS:", len(sin_rostro))
print("ROSTROS DETECTADOS:", len(resultados))

if not tabla.empty:
    display(tabla)

    correctos = (
        tabla["Prediccion"] == "Desconocido"
    ).sum()

    incorrectos = len(tabla) - correctos

    tasa = correctos / len(tabla) * 100

    print("\nRESUMEN")
    print("Clasificados como Desconocido:", correctos)
    print("Identificados como conocidos:", incorrectos)
    print(f"Tasa de rechazo: {tasa:.2f}%")

else:
    print("No se detectaron rostros.")

print("\nFotografías sin rostros:", sin_rostro)
print("Fotografías no leídas:", no_leidas)

# Guardar los resultados
tabla.to_csv(
    "evaluacion_desconocidos.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nResultados guardados.")


FOTOGRAFÍAS ANALIZADAS: 10
FOTOGRAFÍAS NO LEÍDAS: 0
FOTOGRAFÍAS SIN ROSTROS: 0
ROSTROS DETECTADOS: 27


,Fotografia,Rostro,Prediccion,Distancia,Margen
0,imagen_pruebaSEGMENT.jpg,1,Desconocido,0.7047,0.0004
1,images (1).jpg,1,Desconocido,0.6865,0.0199
2,images (2).jpg,1,Desconocido,0.6407,0.0260
3,images (3).jpg,1,Desconocido,0.6254,0.0384
4,images (4).jpg,1,Desconocido,0.7421,0.0210
5,images (5).jpg,1,Desconocido,0.6291,0.0463
6,images (6).jpg,1,Desconocido,0.4956,0.1536
7,images (6).jpg,2,Desconocido,0.6329,0.0180
8,images (6).jpg,3,Desconocido,0.5698,0.0680
9,images (6).jpg,4,Desconocido,0.6586,0.0657



RESUMEN
Clasificados como Desconocido: 27
Identificados como conocidos: 0
Tasa de rechazo: 100.00%

Fotografías sin rostros: []
Fotografías no leídas: []

Resultados guardados.



## Análisis de los resultados

Durante la evaluación se procesaron 10 fotografías de personas no registradas en la galería de referencia, obteniéndose un total de 27 detecciones faciales.

Los 27 rostros fueron clasificados correctamente con la etiqueta **Desconocido**, sin registrar asignaciones incorrectas a las ocho identidades conocidas.

La tasa de rechazo correcto sobre los rostros detectados fue del **100 %**.
